In [1]:
from azureml.core import Workspace, Dataset, Datastore

# اتصال به Workspace
subscription_id = 'f8c5aac3-29fc-4387-858a-1f61722fb57a'
resource_group = 'forskerpl-n0ybkr-rg'
workspace_name = 'forskerpl-n0ybkr-mlw'

ws = Workspace(subscription_id=subscription_id,
               resource_group=resource_group,
               workspace_name=workspace_name)

# گرفتن datastore
datastore = Datastore.get(ws, "researcher_data")

# خواندن همه فایل‌های parquet در مسیر مشخص
dataset = Dataset.Tabular.from_parquet_files(
    path=[(datastore, 'Zahra/012026/Data/MEDS_MDP/data/tuning/*.parquet')]    #     Zahra/Data-07-2025/MDP/MEDS_811/data/train
)

# تبدیل به pandas DataFrame
df = dataset.to_pandas_dataframe()
df.head(15)


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()


Resolving access token for scope "https://storage.azure.com/.default" using identity of type "MANAGED".
Getting data access token with Assigned Identity (client_id=clientid) and endpoint type based on configuration
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,subject_id,time,code,numeric_value
0,30,1990-04-12 00:00:00,DOB,NaN
1,30,2016-12-30 00:00:00,D/DS934,NaN
2,30,2016-12-30 20:51:00,ADMISSION_ADT,NaN
3,30,2016-12-30 20:51:00,MOVE_ADT,NaN
4,30,2016-12-30 20:51:00,^AFSNIT_ADT/,NaN
5,30,2016-12-30 20:56:00,P/UXRG40,NaN
6,30,2016-12-30 21:17:00,P/BLPA80,NaN
7,30,NaT,GENDER//Mand,NaN
8,195,2013-05-25 00:00:00,DOB,NaN
9,195,2018-09-20 00:00:00,D/DS014A,NaN


In [2]:
len(df)

56288629

In [3]:
import pandas as pd

# فرض بر این که فایل CSV رو داری
# df = pd.read_csv("your_file.csv")  # یا مستقیم اگر DataFrame آماده‌ست، نیازی نیست

# دسته‌بندی هر subject_id بر اساس وجود کدهای مختلف
m_patients = df[df['code'].str.startswith('M/', na=False)]['subject_id'].unique()
p_patients = df[df['code'].str.startswith('P/', na=False)]['subject_id'].unique()
d_patients = df[df['code'].str.startswith('D/', na=False)]['subject_id'].unique()
s_patients = df[df['code'].str.startswith('S/', na=False)]['subject_id'].unique()

# کل بیماران منحصربه‌فرد
all_patients = df['subject_id'].unique()

# نمایش آمار
print(f"Whole Patients: {len(all_patients)}")
print(f"The patients has M-medication Codes: {len(m_patients)}")
print(f"The patients has D-diagnosis Codes: {len(d_patients)}")
print(f"The patients has P-Procedure Codes: {len(p_patients)}")
print(f"The patients has S-SKS Codes: {len(s_patients)}")


Whole Patients: 221803
The patients has M-medication Codes: 150771
The patients has D-diagnosis Codes: 221755
The patients has P-Procedure Codes: 212751
The patients has S-SKS Codes: 0


In [4]:
subject_counts = df['subject_id'].value_counts()


In [5]:
subject_counts

684625     56440
132804     49787
1968194    49683
856501     49610
1394370    46260
           ...  
422180         3
1640155        3
1317921        3
1508617        3
1831164        2
Name: subject_id, Length: 221803, dtype: int64

In [6]:
p_Num = df[df['code'].str.startswith('P/', na=False)]

In [7]:
p_Num

,subject_id,time,code,numeric_value
5,30,2016-12-30 20:56:00,P/UXRG40,NaN
6,30,2016-12-30 21:17:00,P/BLPA80,NaN
13,195,2018-09-20 17:50:00,P/BNPA00,NaN
14,195,2018-09-20 17:50:00,P/BNPA82,NaN
15,195,2018-09-20 17:50:00,P/BPNA90,NaN
...,...,...,...,...
56288622,2217994,2024-04-23 10:59:00,P/BVAA34A,NaN
56288624,2217994,2024-04-24 12:59:00,P/BRKP1,NaN
56288625,2217994,2024-04-24 12:59:00,P/BVAA34A,NaN
56288626,2217994,2024-05-08 07:51:00,P/BVAA34A,NaN


In [8]:
import pandas as pd

# پیدا کردن سطرهایی که فقط codeهای نوع /P دارن
only_p = df[df['code'].str.startswith('P/', na=False)]

# بیماران با فقط /P کد
subject_ids_only_p = only_p['subject_id'].unique()

# حالا بیماران با codeهای غیر از /P
not_p = df[~df['code'].str.startswith('P/', na=False)]
subject_ids_with_non_p = set(not_p['subject_id'].unique())

# حذف بیمارانی که فقط /P دارن
only_p_ids_to_exclude = [sid for sid in subject_ids_only_p if sid not in subject_ids_with_non_p]

print("Number of patients with only porcedure code: ", only_p_ids_to_exclude)


Number of patients with only porcedure code:  []


In [9]:
df_filtered = df[~df['code'].str.startswith('P/', na=False)]

In [10]:
subject_counts_MDS = df_filtered['subject_id'].value_counts()

In [11]:
subject_counts_MDS

684625     53884
132804     49438
1968194    47853
856501     47594
1394370    45693
           ...  
219083         2
907736         2
2138700        2
964425         2
1869208        2
Name: subject_id, Length: 221803, dtype: int64

In [12]:
subject_counts_df = subject_counts.reset_index()
subject_counts_df.columns = ['subject_id', 'original_count']

subject_counts_MDS_df = subject_counts_MDS.reset_index()
subject_counts_MDS_df.columns = ['subject_id', 'new_count']


In [13]:
import pandas as pd
comparison_df = pd.merge(subject_counts_df, subject_counts_MDS_df, on='subject_id', how='outer')


In [14]:
comparison_df['difference'] =  comparison_df['original_count'] - comparison_df['new_count']


In [15]:
comparison_df = comparison_df.sort_values(by='difference', ascending=False)


In [16]:
comparison_df

,subject_id,original_count,new_count,difference
21,2109519,20019,16394,3625
24,1664254,19589,16503,3086
0,684625,56440,53884,2556
59,833674,14320,11842,2478
183,2088629,9259,6854,2405
...,...,...,...,...
214030,2132862,7,7,0
197622,901546,15,15,0
179723,2139380,21,21,0
214036,1305973,7,7,0


In [17]:
unchanged_count = (comparison_df['difference'] == 0).sum()
print("unchanged_count", unchanged_count)


unchanged_count 9052


In [18]:
comparison_df['abs_diff'] = comparison_df['difference'].abs()
most_changed = comparison_df.sort_values(by='abs_diff', ascending=False)


In [19]:
print(most_changed.head(10))


      subject_id  original_count  new_count  difference  abs_diff
21       2109519           20019      16394        3625      3625
24       1664254           19589      16503        3086      3086
0         684625           56440      53884        2556      2556
59        833674           14320      11842        2478      2478
183      2088629            9259       6854        2405      2405
54       1040286           14640      12362        2278      2278
45        261787           14957      12710        2247      2247
1728     1235924            3351       1214        2137      2137
98        180827           11855       9750        2105      2105
62        172103           14227      12153        2074      2074


In [20]:
changed_df = comparison_df[comparison_df['difference'] != 0]
min_new_count = changed_df['new_count'].min()
lowest_new_count_patients = changed_df[changed_df['new_count'] == min_new_count]


In [21]:
lowest_new_count_patients = lowest_new_count_patients.rename(
    columns={
        'original_count': 'MDP codes',
        'new_count': 'MD codes'
    }
)


In [22]:
lowest_new_count_patients

,subject_id,MDP codes,MD codes,difference,abs_diff
173594,1181764,24,2,22,22
190631,43244,17,2,15,15
200216,1540904,14,2,12,12
207894,148117,10,2,8,8
209430,620878,10,2,8,8
210855,177233,9,2,7,7
210285,242591,9,2,7,7
211452,1085023,8,2,6,6
212268,1869208,8,2,6,6
212038,899005,8,2,6,6


.str.upper() شرط را case-insensitive می‌کند

بل از فیلتر، ستون را به pd.StringDtype() تبدیل می‌کند؛ این کار رفتار .str را پایدار و قابل پیش‌بینی می‌کند (<NA> به‌جای NaN)



In [23]:
# کل بیماران منحصربه‌فرد
all_patients = df_filtered['subject_id'].unique()
len(all_patients)
# نمایش آمار

221803

In [24]:
len(df_filtered)

45705092

In [25]:
import pandas as pd
import numpy as np

# امن‌تر: اگر code نال یا غیررشته‌ای بود اذیت نکنه
df['code'] = df['code'].astype('string')
df_filtered = df[~df['code'].str.upper().str.startswith('P/', na=False)].copy()
print("kept rows MDP:", len(df_filtered), " / total:", len(df))


kept rows MDP: 45705092  / total: 56288629


In [26]:
import pandas as pd
import numpy as np

# اطمینان از نوع‌ها (برای خروجی تمیز و بدون خطا)
df_filtered = df_filtered.copy()
df_filtered['subject_id']    = pd.to_numeric(df_filtered['subject_id'], errors='coerce').astype('Int64')
df_filtered['numeric_value'] = pd.to_numeric(df_filtered['numeric_value'], errors='coerce').astype('float32')
df_filtered['time']          = pd.to_datetime(df_filtered['time'], errors='coerce', utc=False)

# ردیف‌های بدون subject_id را حذف کنیم (نمی‌توان شارد کرد)
df_filtered = df_filtered.dropna(subset=['subject_id']).copy()
df_filtered['subject_id'] = df_filtered['subject_id'].astype('int64')

# ۳۶ شارد: هر بیمار فقط در یک فایل (mod 36)
#N_SHARDS = 45
#df_filtered['__shard__'] = (df_filtered['subject_id'] % N_SHARDS).astype('int16')

#print("rows to write:", len(df_filtered))


In [27]:
import numpy as np
import os

N_SHARDS = 5   #45 for Whole # 36 when we have split
OUT_DIR = "./_tuningMDP_withoutP_sharded"
os.makedirs(OUT_DIR, exist_ok=True)

cols_out = ['subject_id', 'time', 'code', 'numeric_value']

# فرض: subject_id قبلاً int64 شده و NaNها حذف شده‌اند (طبق سلول قبلی‌ات)
sid_mod = (df_filtered['subject_id'].to_numpy(dtype=np.int64, copy=False) % N_SHARDS)

written = 0
for k in range(N_SHARDS):
    mask = (sid_mod == k)                 # بدون ستون اضافی، فقط یک آرایهٔ NumPy
    part = df_filtered.loc[mask, cols_out].sort_values(['subject_id','time'])
    # اگر می‌خوای حتماً ۳۶ فایل 0..35 داشته باشی حتی اگه خالی باشن:
    # if part.empty:
    #     part = part.iloc[0:0]  # فایل صفر-سطر با همان ستون‌ها
    part.to_parquet(os.path.join(OUT_DIR, f"{k}.parquet"),
                    engine="pyarrow", compression="snappy", index=False)
    written += 1

print(f"Done. wrote {written} parquet files into {OUT_DIR}")


Done. wrote 5 parquet files into ./_tuningMDP_withoutP_sharded


In [28]:
import pyarrow.parquet as pq

OUT_DIR = "./_tuningMDP_withoutP_sharded"

def parquet_num_rows(path):
    pf = pq.ParquetFile(path)
    md = pf.metadata
    return sum(md.row_group(i).num_rows for i in range(md.num_row_groups))

total = 0
for f in sorted(p for p in os.listdir(OUT_DIR) if p.endswith('.parquet')):
    n = parquet_num_rows(os.path.join(OUT_DIR, f))
    total += n
    print(f, "rows:", n)
print("TOTAL rows:", total)


0.parquet rows: 9108892
1.parquet rows: 9093752
2.parquet rows: 9374059
3.parquet rows: 9150531
4.parquet rows: 8977858
TOTAL rows: 45705092


In [29]:
from azureml.data.datapath import DataPath
from azureml.data.dataset_factory import FileDatasetFactory
import os

OUT_DIR = "./_tuningMDP_withoutP_sharded"
DST_PREFIX = "Zahra/012026/Data/MEDS_MD/data/tuning"  #held_out"  # بدون اسلشِ اول
''

# اطمینان: پوشه خروجی وجود دارد و فایل parquet داخلش هست
print("Local files to upload:", len([f for f in os.listdir(OUT_DIR) if f.endswith(".parquet")]))

# مقصد روی Datastore
target = DataPath(datastore, DST_PREFIX)

# آپلود همه محتویات OUT_DIR به DST_PREFIX
_ = FileDatasetFactory.upload_directory(
    src_dir=OUT_DIR,
    target=target,
    overwrite=True,
    show_progress=True,
)
print(f"Uploaded to datastore path: {DST_PREFIX}")


Local files to upload: 5
Validating arguments.
Arguments validated.
'overwrite' is set to True. Any file already present in the target will be overwritten.
Uploading files from '/mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_tuningMDP_withoutP_sharded' to 'Zahra/012026/Data/MEDS_MD/data/tuning'
Copying 5 files with concurrency set to 5
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_tuningMDP_withoutP_sharded/4.parquet, file 1 out of 5. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/012026/Data/MEDS_MD/data/tuning/4.parquet
Copied /mnt/batch/tasks/shared/LS_root/mounts/clusters/zahra/code/Users/zahra.sobhaninia/Corebehrt_CV/CreateIdenticalData/_tuningMDP_withoutP_sharded/2.parquet, file 2 out of 5. Destination path: https://forskerpln0ybkrdls01.dfs.core.windows.net/researcher-data/Zahra/012026/Data/MED

In [30]:
paths = Dataset.File.from_files(path=[(datastore, f"{DST_PREFIX}/*.parquet")]).to_path()
print("Found in datastore:", len(paths))
print(paths[:20])


{'infer_column_types': 'False', 'activity': 'to_path'}
{'infer_column_types': 'False', 'activity': 'to_path', 'activityApp': 'FileDataset'}
Found in datastore: 5
['/0.parquet', '/1.parquet', '/2.parquet', '/3.parquet', '/4.parquet']
